# Sistema RAG — Asistente Conversacional Clínico

**TFM:** Sistema de Apoyo a la Decisión Clínica en Oncología Pediátrica
**Autor:** Alonso Castañón González

**Objetivo de este notebook:** Construir el sistema RAG (Retrieval-Augmented Generation) que permite a oncólogos y familias consultar en lenguaje natural información sobre osteosarcoma y sarcoma de Ewing pediátrico, combinando conocimiento general con la predicción específica del modelo XGBoost entrenado en `03_modelo_ML.ipynb`.

**Arquitectura:** LangGraph con 3 nodos (`retrieve` → `postfiltering` → `generate`), embeddings locales gratuitos (HuggingFace) e indexado en Qdrant (Docker local), con generación mediante la API de OpenAI.

In [8]:
import os
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from langchain_huggingface import HuggingFaceEmbeddings
from qdrant_client import QdrantClient
import requests
from bs4 import BeautifulSoup
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document
from collections import Counter
from langchain_qdrant import QdrantVectorStore
from qdrant_client.http.models import Distance, VectorParams

## 1) Setup

Se cargan las credenciales desde `.env` y se verifica la conexión a Qdrant, HuggingFace y OpenAI antes de empezar.

In [11]:
load_dotenv()

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
assert OPENAI_API_KEY is not None, "No se encontró OPENAI_API_KEY en el archivo .env"

print("Variables de entorno cargadas correctamente")

QDRANT_HOST = "localhost"
QDRANT_PORT = 6333

qdrant_client = QdrantClient(host=QDRANT_HOST, port=QDRANT_PORT)

print("Colecciones existentes en Qdrant:", qdrant_client.get_collections())

embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2",
    model_kwargs={'device': 'cpu'}
)

test_embedding = embeddings.embed_query("prueba de conexión")
print(f"Embedding generado localmente, dimensión: {len(test_embedding)}")

llm = ChatOpenAI(model="gpt-4o-mini", api_key=OPENAI_API_KEY, temperature=0)

test_response = llm.invoke("Responde solo con la palabra 'OK' si me recibes.")
print(f"Respuesta del LLM: {test_response.content}")

Variables de entorno cargadas correctamente
Colecciones existentes en Qdrant: collections=[CollectionDescription(name='tfm_oncologia_pediatrica')]


modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/3.89k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer.json: reconstructing file:   0%|          |  0.00B / 9.08MB            

tokenizer.json: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Embedding generado localmente, dimensión: 384
Respuesta del LLM: OK


## 2) Documentos y metadatos

Se descargan los 8 documentos NCI mediante scraping, extrayendo el contenido principal de cada página (recortando el pie de página administrativo común a las fichas PDQ: permisos, citación, información de ensayos clínicos) y etiquetando cada uno por enfermedad y tipo. Estos documentos serán los que alimenten al RAG y de los cuales surgirán sus respuestas. Es por eso que se han buscado documentos de varios tipos, más específicos sobre los tratamientos del osteosarcoma y de ewing, o más generales de cara a la explicación a las familias. 

In [3]:
import requests
from bs4 import BeautifulSoup

# Encabezados a partir de los cuales se recorta el contenido
MARKERS = [
    "About This PDQ Summary", "About PDQ", "Purpose of This Summary",
    "Reviewers and Updates", "Clinical Trial Information",
    "Permission to Use This Summary", "Disclaimer", "Contact Us",
    "Related resources"
]

def scrape_nci_page(url):
    res = requests.get(url, headers={"User-Agent": "Mozilla/5.0"})
    res.raise_for_status()
    soup = BeautifulSoup(res.text, "html.parser")

    main = soup.find("main") or soup.find(id="main-content")
    if main is None:
        raise ValueError(f"No se encontró el contenido principal en {url}")

    title = soup.find("h1").get_text(strip=True) if soup.find("h1") else url

    parts = []
    for el in main.find_all(["h2", "h3", "h4", "p", "li"]):
        text = el.get_text(strip=True)
        if not text:
            continue
        # Solo cortar en encabezados reales, no en enlaces del índice "On This Page"
        if el.name in ["h2", "h3", "h4"] and any(marker in text for marker in MARKERS):
            break
        parts.append(text)

    return {"title": title, "url": url, "text": "\n".join(parts)}


documentos_nci = [
    {"url": "https://www.cancer.gov/types/bone/patient/osteosarcoma-treatment-pdq",
     "enfermedad": "osteosarcoma", "tipo": "tratamiento"},
    {"url": "https://www.cancer.gov/types/bone/patient/ewing-treatment-pdq",
     "enfermedad": "ewing", "tipo": "tratamiento"},
    {"url": "https://www.cancer.gov/about-cancer/treatment/types/chemotherapy",
     "enfermedad": "general", "tipo": "tratamiento"},
    {"url": "https://www.cancer.gov/about-cancer/treatment/types/surgery",
     "enfermedad": "general", "tipo": "tratamiento"},
    {"url": "https://www.cancer.gov/about-cancer/treatment/types/radiation-therapy",
     "enfermedad": "general", "tipo": "tratamiento"},
    {"url": "https://www.cancer.gov/about-cancer/coping/caregiver-support/parents",
     "enfermedad": "general", "tipo": "apoyo_emocional"},
    {"url": "https://www.cancer.gov/about-cancer/understanding/what-is-cancer",
 "enfermedad": "general", "tipo": "conceptos_basicos"},
    {"url": "https://www.cancer.gov/types/childhood-cancers",
     "enfermedad": "general", "tipo": "informativo"},
]

In [4]:
documentos = []
for doc in documentos_nci:
    scraped = scrape_nci_page(doc["url"])
    scraped["enfermedad"] = doc["enfermedad"]
    scraped["tipo"] = doc["tipo"]
    documentos.append(scraped)
    print(f"OK - {scraped['title']} — {len(scraped['text'])} caracteres")

OK - Osteosarcoma Treatment (PDQ®)–Patient Version — 24655 caracteres
OK - Ewing Sarcoma Treatment (PDQ®)–Patient Version — 27690 caracteres
OK - Chemotherapy to Treat Cancer — 10991 caracteres
OK - Surgery to Treat Cancer — 15139 caracteres
OK - Radiation Therapy to Treat Cancer — 10886 caracteres
OK - Support for Families: Childhood Cancer — 24036 caracteres
OK - What Is Cancer? — 18242 caracteres
OK - Childhood Cancers — 9558 caracteres


## 3) Chunking

Se divide cada documento en fragmentos manejables para el retrieval. Se usa `chunk_size=1000`, ya que en principio los documentos NCI son narrativos y se benefician de fragmentos algo más largos para mantener contexto coherente También con `chunk_overlap=200` y conservando los metadatos (`enfermedad`, `tipo`, `title`, `url`) en cada fragmento.

In [7]:
splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200,
    separators=["\n\n", "\n", ". ", " ", ""]
)

split_docs = []
for doc in documentos:
    chunks = splitter.split_text(doc["text"])
    for chunk in chunks:
        if chunk.strip():
            split_docs.append(
                Document(
                    page_content=chunk,
                    metadata={
                        "title": doc["title"],
                        "url": doc["url"],
                        "enfermedad": doc["enfermedad"],
                        "tipo": doc["tipo"],
                    }
                )
            )

print(f"Total de fragmentos generados: {len(split_docs)}")

conteo = Counter(d.metadata["title"] for d in split_docs)
for titulo, n in conteo.items():
    print(f"  {titulo}: {n} fragmentos")

Total de fragmentos generados: 183
  Osteosarcoma Treatment (PDQ®)–Patient Version: 33 fragmentos
  Ewing Sarcoma Treatment (PDQ®)–Patient Version: 36 fragmentos
  Chemotherapy to Treat Cancer: 15 fragmentos
  Surgery to Treat Cancer: 19 fragmentos
  Radiation Therapy to Treat Cancer: 14 fragmentos
  Support for Families: Childhood Cancer: 30 fragmentos
  What Is Cancer?: 23 fragmentos
  Childhood Cancers: 13 fragmentos


## 4) Indexado en Qdrant

Se crea la colección en Qdrant (recreándola si ya existiera, para evitar duplicados al reejecutar el notebook) y se suben los 183 fragmentos con sus embeddings y metadatos.

In [12]:
COLLECTION_NAME = "tfm_oncologia_pediatrica"

# Recrear la colección para evitar duplicados
if qdrant_client.collection_exists(COLLECTION_NAME):
    qdrant_client.delete_collection(COLLECTION_NAME)

qdrant_client.create_collection(
    collection_name=COLLECTION_NAME,
    vectors_config=VectorParams(size=384, distance=Distance.COSINE)  # 384 = dimensión de MiniLM-L6-v2
)

qdrant = QdrantVectorStore(
    client=qdrant_client,
    collection_name=COLLECTION_NAME,
    embedding=embeddings,
)

qdrant.add_documents(split_docs)

print(f"Colección '{COLLECTION_NAME}' creada con {len(split_docs)} fragmentos indexados")

Colección 'tfm_oncologia_pediatrica' creada con 183 fragmentos indexados


In [13]:
resultados_prueba = qdrant.similarity_search("¿Qué es la quimioterapia neoadyuvante?", k=3)

for r in resultados_prueba:
    print(f"[{r.metadata['enfermedad']} / {r.metadata['tipo']}] {r.metadata['title']}")
    print(r.page_content[:200], "...\n")

[osteosarcoma / tratamiento] Osteosarcoma Treatment (PDQ®)–Patient Version
Chemotherapy
Chemotherapy (also called chemo) uses drugs to stop the growth of cancer cells. Chemotherapy either kills the cancer cells or stops them from dividing.   Chemotherapy may be given alone o ...

[general / tratamiento] Chemotherapy to Treat Cancer
Chemotherapy works against cancer by killing fast-growing cancer cells.
Credit: National Cancer Institute
Chemotherapy(also called chemo) is a type of cancer treatment that uses drugs to kill cancerce ...

[osteosarcoma / tratamiento] Osteosarcoma Treatment (PDQ®)–Patient Version
For tumors  that have recurred twice, treatment may include:surgery to remove the cancer and chemotherapychemotherapy alone
surgery to remove the cancer and chemotherapy
chemotherapy alone
Learn more  ...

